In [130]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [22]:
df = pd.read_parquet('../data/processed/df_clean.parquet')

In [26]:
df[['run_up_distance', 'race_type', 'purse', 'post_time',
       'weight_carried', 'jockey', 'odds', 'position_at_finish',
       'distance_id_m', 'run_up_distance_m', 'horse_pk', 'rid', 'horse_id',
       'horse_name', 'win']]

,run_up_distance,race_type,purse,post_time,weight_carried,jockey,odds,position_at_finish,distance_id_m,run_up_distance_m,horse_pk,rid,horse_id,horse_name,win
0,36,AOC,80000.0,1220,123,Dylan Davis,130,2,1307.59,10.97,AQU_2019-01-01_1_1,AQU_2019-01-01_1,1,Sounds Delicious,0
1,36,AOC,80000.0,1220,123,Dylan Davis,130,2,1307.59,10.97,AQU_2019-01-01_1_1,AQU_2019-01-01_1,1,Sounds Delicious,0
2,36,AOC,80000.0,1220,123,Dylan Davis,130,2,1307.59,10.97,AQU_2019-01-01_1_1,AQU_2019-01-01_1,1,Sounds Delicious,0
3,36,AOC,80000.0,1220,123,Dylan Davis,130,2,1307.59,10.97,AQU_2019-01-01_1_1,AQU_2019-01-01_1,1,Sounds Delicious,0
4,36,AOC,80000.0,1220,123,Dylan Davis,130,2,1307.59,10.97,AQU_2019-01-01_1_1,AQU_2019-01-01_1,1,Sounds Delicious,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5065652,22,STK,250000.0,507,120,John R. Velazquez,265,5,1709.93,6.71,SAR_2019-09-02_9_7,SAR_2019-09-02_9,2941,Olympico (FR),0
5065653,22,STK,250000.0,507,120,John R. Velazquez,265,5,1709.93,6.71,SAR_2019-09-02_9_7,SAR_2019-09-02_9,2941,Olympico (FR),0
5065654,22,STK,250000.0,507,120,John R. Velazquez,265,5,1709.93,6.71,SAR_2019-09-02_9_7,SAR_2019-09-02_9,2941,Olympico (FR),0
5065655,22,STK,250000.0,507,120,John R. Velazquez,265,5,1709.93,6.71,SAR_2019-09-02_9_7,SAR_2019-09-02_9,2941,Olympico (FR),0


In [94]:
df

,track_id,race_date,race_number,program_number,trakus_index,latitude,longitude,distance_id,course_type,track_condition,...,prev_time_seconds,is_first_obs,distance_m,cumulative_distance_m,cum_race_distance_m,speed_kmh,position_rank,pctComplete,race_max_distance_m,Segment
0,AQU,2019-01-01,1,1,1,40.669401,-73.829205,650,Dirt,MY,...,NaN,True,0.000000,0.000000,-10.970000,0.000000,1,0.000000,1454.256448,Q1
1,AQU,2019-01-01,1,1,2,40.669405,-73.829203,650,Dirt,MY,...,0.25,False,0.529150,0.529150,-10.440850,7.619765,1,0.000364,1454.256448,Q1
2,AQU,2019-01-01,1,1,3,40.669411,-73.829200,650,Dirt,MY,...,0.50,False,0.734889,1.264039,-9.705961,10.582402,1,0.000869,1454.256448,Q1
3,AQU,2019-01-01,1,1,4,40.669421,-73.829196,650,Dirt,MY,...,0.75,False,1.110307,2.374346,-8.595654,15.988421,1,0.001633,1454.256448,Q1
4,AQU,2019-01-01,1,1,5,40.669433,-73.829190,650,Dirt,MY,...,1.00,False,1.379717,3.754063,-7.215937,19.867919,1,0.002581,1454.256448,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5065652,SAR,2019-09-02,9,7,373,43.071943,-73.770255,850,Turf,SF,...,93.00,False,4.759391,1722.983478,1716.273478,68.535235,3,0.989321,1741.581552,Q4
5065653,SAR,2019-09-02,9,7,374,43.071924,-73.770305,850,Turf,SF,...,93.25,False,4.616698,1727.600176,1720.890176,66.480445,3,0.991972,1741.581552,Q4
5065654,SAR,2019-09-02,9,7,375,43.071905,-73.770357,850,Turf,SF,...,93.50,False,4.709587,1732.309763,1725.599763,67.818053,2,0.994676,1741.581552,Q4
5065655,SAR,2019-09-02,9,7,376,43.071885,-73.770409,850,Turf,SF,...,93.75,False,4.706746,1737.016509,1730.306509,67.777149,2,0.997379,1741.581552,Q4


In [52]:
testDF = df[df['rid']=='AQU_2019-01-01_1']

In [59]:
testDF[testDF['trakus_index']==testDF['trakus_index'].max()][['trakus_index','distance_id','cumulative_distance_m','distance_id_m']]

,trakus_index,distance_id,cumulative_distance_m,distance_id_m
315,316,650,1440.583247,1307.59
631,316,650,1424.462864,1307.59
947,316,650,1420.315348,1307.59
1263,316,650,1424.354586,1307.59
1579,316,650,1454.256448,1307.59


In [66]:
df['race_max_distance_m'] = df.groupby('rid')['cumulative_distance_m'].transform('max')

In [82]:
df['pctComplete'] = (df['cumulative_distance_m'] / df['race_max_distance_m'])

In [80]:
varianceDF = df.groupby('rid')[['distance_id_m','race_max_distance_m']].first().sample(20)

varianceDF['Variance %'] = round(100 * (1 - (varianceDF['distance_id_m'] / varianceDF['race_max_distance_m'])),2)

varianceDF

,distance_id_m,race_max_distance_m,Variance %
rid,,,
AQU_2019-04-14_4,1207.01,1356.716883,11.03
BEL_2019-09-20_6,1307.59,1489.556855,12.22
AQU_2019-12-30_3,1609.34,1873.340812,14.09
SAR_2019-08-21_7,1408.18,1439.450760,2.17
AQU_2019-01-18_3,1207.01,1356.329884,11.01
AQU_2019-01-27_9,1609.34,1869.516715,13.92
AQU_2019-01-26_7,1609.34,1764.353204,8.79
SAR_2019-07-21_3,1810.51,1874.555443,3.42
BEL_2019-09-18_6,1207.01,1380.761057,12.58


In [91]:
conditions = [
    df['pctComplete'] < 0.25,
    df['pctComplete'] < 0.50,
    df['pctComplete'] < 0.75,
    df['pctComplete'] <= 1.00
]
choices = ['Q1', 'Q2', 'Q3', 'Q4']

df['Segment'] = np.select(conditions, choices)

In [100]:
df[['track_id','distance_id','course_type','track_condition','race_type','position_at_finish','win','pctComplete','speed_kmh','position_rank','Segment','speed_Q1','speed_Q2','speed_Q3','speed_Q4']]

,track_id,distance_id,course_type,track_condition,race_type,position_at_finish,win,pctComplete,speed_kmh,position_rank,Segment,speed_Q1,speed_Q2,speed_Q3,speed_Q4
0,AQU,650,Dirt,MY,AOC,2,0,0.000000,0.000000,1,Q1,65.428845,NaN,NaN,NaN
1,AQU,650,Dirt,MY,AOC,2,0,0.000364,7.619765,1,Q1,65.428845,NaN,NaN,NaN
2,AQU,650,Dirt,MY,AOC,2,0,0.000869,10.582402,1,Q1,65.428845,NaN,NaN,NaN
3,AQU,650,Dirt,MY,AOC,2,0,0.001633,15.988421,1,Q1,65.428845,NaN,NaN,NaN
4,AQU,650,Dirt,MY,AOC,2,0,0.002581,19.867919,1,Q1,65.428845,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5065652,SAR,850,Turf,SF,STK,5,0,0.989321,68.535235,3,Q4,NaN,NaN,NaN,71.158647
5065653,SAR,850,Turf,SF,STK,5,0,0.991972,66.480445,3,Q4,NaN,NaN,NaN,71.158647
5065654,SAR,850,Turf,SF,STK,5,0,0.994676,67.818053,2,Q4,NaN,NaN,NaN,71.158647
5065655,SAR,850,Turf,SF,STK,5,0,0.997379,67.777149,2,Q4,NaN,NaN,NaN,71.158647


In [111]:
df['speed_Q1'] = df[df['Segment'] == 'Q1'].groupby('horse_pk')['speed_kmh'].transform('mean')
df['speed_Q2'] = df[df['Segment'] == 'Q2'].groupby('horse_pk')['speed_kmh'].transform('mean')
df['speed_Q3'] = df[df['Segment'] == 'Q3'].groupby('horse_pk')['speed_kmh'].transform('mean')
df['speed_Q4'] = df[df['Segment'] == 'Q4'].groupby('horse_pk')['speed_kmh'].transform('mean')

df['pos_Q1'] = df[df['Segment'] == 'Q1'].groupby('horse_pk')['position_rank'].transform('median')
df['pos_Q2'] = df[df['Segment'] == 'Q2'].groupby('horse_pk')['position_rank'].transform('median')
df['pos_Q3'] = df[df['Segment'] == 'Q3'].groupby('horse_pk')['position_rank'].transform('median')
df['pos_Q4'] = df[df['Segment'] == 'Q4'].groupby('horse_pk')['position_rank'].transform('median')

In [127]:
finalDF = df.groupby('horse_pk').agg({
    'track_id':'first',
    'distance_id':'first',
    'course_type':'first',
    'track_condition':'first',
    'race_type':'first',
    'win':'first',
    'speed_Q1':'first',
    'speed_Q2':'first',
    'speed_Q3':'first',
    'speed_Q4':'first',
    'pos_Q1':'first',
    'pos_Q2':'first',
    'pos_Q3':'first',
    'pos_Q4':'first',
    })

In [128]:
finalDF = finalDF.dropna(subset=['pos_Q1', 'pos_Q2', 'pos_Q3', 'pos_Q4'])

# This accounts for horses that fell or did not finish for another reason (less than 0.4% of the data)

In [139]:
def winClassifier(df):
    predictor_cols = ['speed_Q1', 'speed_Q2', 'speed_Q3', 'speed_Q4',
                    'pos_Q1', 'pos_Q2', 'pos_Q3', 'pos_Q4']
    df = df.dropna(subset=predictor_cols)

    X = df[predictor_cols]
    y = df['win']


    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )


    model = LogisticRegression(max_iter=1000)


    model.fit(X_train, y_train)


    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]


    coef_df = pd.DataFrame({
        'Feature': predictor_cols,
        'Coefficient': model.coef_[0]
    }).sort_values(by='Coefficient', ascending=False)

    return coef_df

In [146]:


def strategy_guide(df, coef_df):

    # Strategy logic
    def strategy_tip(row):
        f, coef = row['Feature'], row['Coefficient']
        quarter = f[-2:]  # Q1, Q2, Q3, Q4
        impact = abs(coef)
        
        # Position-based features
        if 'pos' in f:
            if coef < -0.75:
                return (f"🏁 **{quarter}: Crucial Positioning Phase** — "
                        f"your odds of winning drop sharply with every position lost. "
                        f"Make sure you're contesting the lead entering this segment.")
            elif coef < -0.25:
                return (f"⚠️ **{quarter}: Positional Pressure** — "
                        f"falling more than a few lengths back reduces your win potential. "
                        f"Stay engaged with the leaders and prepare to respond.")
            elif coef > 0.25:
                return (f"🧠 **{quarter}: Tactical Hold-Back Opportunity** — "
                        f"jockeys positioned just off the pace here tend to set up successful closing runs. "
                        f"Avoid over-committing too early.")
            else:
                return (f"ℹ️ **{quarter}: Flexible Positioning** — "
                        f"no strong positional impact observed. Focus on conserving energy and tracking the pace.")
        
        # Speed-based features
        elif 'speed' in f:
            if coef > 0.1:
                return (f"🚀 **{quarter}: Momentum Advantage** — "
                        f"faster speeds here have a meaningful impact. If you're well-positioned, consider driving forward to gain tactical edge.")
            elif coef > 0.02:
                return (f"⏩ **{quarter}: Marginal Speed Gain** — "
                        f"increasing pace helps, but only slightly. Push only if it won’t cost you late.")
            elif coef < -0.1:
                return (f"⛔ **{quarter}: Risk of Overexertion** — "
                        f"higher speed in this segment typically backfires. Focus on rhythm and efficiency.")
            elif coef < -0.02:
                return (f"⚠️ **{quarter}: Subtle Fatigue Zone** — "
                        f"be cautious — small increases in speed don’t pay off. Prioritize form and control.")
            else:
                return (f"ℹ️ **{quarter}: Neutral Speed Effect** — "
                        f"no significant impact from changes in speed here. Let race dynamics guide your pacing.")
        
        # Fallback
        else:
            return "No strategy available for this feature."

    # Apply strategy interpretation
    coef_df['Odds Multiplier'] = np.exp(coef_df['Coefficient'])
    coef_df['Strategy Tip'] = coef_df.apply(strategy_tip, axis=1)

    # Reorder and show as strategy guide
    strategy_guide = coef_df.sort_values(by='Feature')[['Feature', 'Coefficient', 'Odds Multiplier', 'Strategy Tip']]

    return strategy_guide['Strategy Tip'] 

strategy_guide(df, winClassifier(df))

4    🏁 In Q1, aim to be in **front position** — fal...
5    🧠 In Q2, being slightly behind may set up a wi...
6    🧠 In Q3, being slightly behind may set up a wi...
7    🏁 In Q4, aim to be in **front position** — fal...
0    ⚠️ In Q1, higher speed slightly reduces win od...
1    🚀 In Q2, increasing speed helps — consider acc...
2    ⚠️ In Q3, higher speed slightly reduces win od...
3    🚀 In Q4, increasing speed helps — consider acc...
Name: Strategy Tip, dtype: object
